# Eulerian analysis vertical shear on horizontal transport
In our work we ignored the effect of the $\boldsymbol{w}_f \frac{\partial d u_f}{\partial z}$ and $\boldsymbol{w}_f \frac{\partial d v_f}{\partial z}$ term on the horizontal transport. Here we compare the magnitude of these term with the other terms in the material derivative in the eulerian framework.   

Note: The dataset of the NWES we used from the model run in Nologin Spain does not provide the hourly vertical data, it did provide the daily vertical data but we did not save this locally. The copernicusmarineservice changed model at 1 jan 2026 for the NWES from the Nologin spain model to the met-office model which means that we no longer have access to data of the model we used for this work. The new model has a much lower vertical resolution (3 meters vs 1 meter). Thus we chose here to do the analysis for the Iberia–Biscay–Ireland (IBI) area instead (https://doi.org/10.48670/moi-00027) which was simulated with the same model as we used in our work (runned by NOW/Nologin Spain). Both models at the moment only have daily data for the vertical velocities. The met-office model does have vertical values in their hourly model but these are always zero execpt once a day. 

In [ ]:
# import needed packages
import sys
sys.path.append("/nethome/4291387/Maxey_Riley_advection/Maxey_Riley_advection/src")
import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import xgcm
import cartopy.crs as ccrs 
import cartopy as cart
import cartopy.feature as cfeature
from datetime import datetime, timedelta
import copernicusmarine  

# login 
copernicusmarine.login(configuration_file_directory="/nethome/4291387/.copernicusmarine")

from analysis_functions import make_PDF, Haversine, make_lognormal_PDF

# plotstyle: 
plt.style.use('../python_style_Meike.mplstyle')

In [ ]:
# stream data (as we can only go 2 years back we not look at seb 2023 but sep 2025)
starttime = datetime(2025,9,1,0,0,0)
runtime = timedelta(days=2)
endtime = starttime + runtime 
ds_w = copernicusmarine.open_dataset("cmems_mod_ibi_phy-wcur_anfc_0.027deg_P1D-m", minimum_depth = 0, maximum_depth= 1.5, start_datetime = starttime, end_datetime =  endtime,coordinates_selection_method= 'inside')
ds_uv = copernicusmarine.open_dataset("cmems_mod_ibi_phy_anfc_0.027deg-3D_P1D-m",minimum_depth = 0, maximum_depth= 2, start_datetime = starttime, end_datetime =  endtime,coordinates_selection_method= 'inside')

In [ ]:
# derivatives 
# grid resolution model
delta_lon = ds_uv.longitude[1]-ds_uv.longitude[0]
delta_lat  = ds_uv.latitude[1]-ds_uv.latitude[0]
delta_t = ds_uv.time[1]-ds_uv.time[0]
delta_t_s = delta_t/np.timedelta64(1, 's') # delta_t in seconds

# define left points grids
lon_left = ds_uv.longitude - 0.5*delta_lon
lon_left=lon_left.rename(longitude="lon_left")
lat_left = ds_uv.latitude - 0.5*delta_lat
lat_left=lat_left.rename(latitude="lat_left")
depth_top = ds_w.depth
depth_top=depth_top.rename(depth="depth_top")
time_left = ds_uv.time-0.5*delta_t
time_left = time_left.rename(time='time_left')

# add left points to coordinates data
ds_uv = ds_uv.assign_coords(lon_left = lon_left)
ds_uv = ds_uv.assign_coords(lat_left = lat_left)
ds_uv = ds_uv.assign_coords(time_left = time_left)
ds_uv = ds_uv.assign_coords(depth_top = depth_top)

# dx at u-point (between lon centers, on face)
dlon_center = ds_uv['longitude'].diff('longitude') * np.pi / 180  # radians
dy_center   = ds_uv['latitude'].diff('latitude') * np.pi / 180  # radians

#calculate differences in meters
R_EARTH = 6371000.0  # meters
ds_uv['dx'] = dlon_center * np.cos(ds_uv['latitude'] * np.pi / 180) * R_EARTH
ds_uv['dy'] = dy_center * R_EARTH

# calculate grid
grid = xgcm.Grid(ds_uv, coords={"lon": {"center": "longitude","left":"lon_left"},
                                "lat":{"center":"latitude","left":"lat_left"},
                                "time":{"center":"time","left":"time_left"},
                                "depth":{"center":"depth","left":"depth_top"}},
                metrics={('lon',): ['dx'],   # distance between X-points in meters
                        ('lat',): ['dy']},   # distance between Y-points in meters
                autoparse_metadata=False,  
                periodic=False)



# calculate derivative fields
dudx = grid.derivative(ds_uv['uo'], 'lon')   # m/s/m
dudx = grid.interp(dudx,'lon',to='center')
dudy = grid.derivative(ds_uv['uo'], 'lat')   # m/s/m
dudy = grid.interp(dudy,'lat',to='center')
dudz = grid.diff(ds_uv['uo'],axis='depth')/ds_w.depth[1].values
dvdx = grid.derivative(ds_uv['vo'], 'lon')   # m/s/m
dvdx = grid.interp(dvdx,'lon',to='center')
dvdy = grid.derivative(ds_uv['vo'], 'lat')   # m/s/m
dvdy = grid.interp(dvdy,'lat',to='center')
dvdz = grid.diff(ds_uv['vo'],axis='depth')/ds_w.depth[1].values


ds_derivative = xr.Dataset(data_vars={'dudx':dudx,
                                      'dudy':dudy,
                                      'dudz':dudz,
                                      'dvdx':dvdx,
                                      'dvdy':dvdy,
                                      'dvdz':dvdz,})

In [ ]:
# product terms in the material derivative
u_dudx = ds_uv.uo.isel(time=0,depth=0) * ds_derivative.dudx.isel(time=0,depth=0)
v_dudy = ds_uv.vo.isel(time=0,depth=0) * ds_derivative.dudy.isel(time=0,depth=0)
w_dudz= ds_w.wo.isel(time=0,depth=0) * ds_derivative.dudz.isel(time=0,depth_top =0)

ratio= np.sqrt(u_dudx**2 + v_dudy**2) / np.abs(w_dudz)


In [ ]:
ds_w.wo.isel(time=0,depth=1)[0:10,0:10].values

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(30,8),subplot_kw={'projection':ccrs.PlateCarree()})
pc1 =axs[0].pcolormesh(u_dudx.longitude,u_dudx.latitude,u_dudx,vmin=-1e-6,vmax=1e-6,cmap='RdBu')
pc2 = axs[1].pcolormesh(v_dudy.longitude,v_dudy.latitude,v_dudy,vmin=-1e-6,vmax=1e-6,cmap='RdBu')
pc3 = axs[2].pcolormesh(w_dudz.longitude,w_dudz.latitude,w_dudz,vmin=-1e-9,vmax=1e-9,cmap='RdBu')
cb1 = fig.colorbar(pc1,extend='both',fraction=0.05)
cb1.set_label('u$\\cdot$du/dx [1/s]', fontsize=18)
cb1.ax.tick_params(labelsize=18) 
cb1.ax.yaxis.offsetText.set_fontsize(18)
cb2 = fig.colorbar(pc2,extend='both',fraction=0.05)
cb2.set_label('v$\\cdot$du/dy [1/s]', fontsize=18)
cb2.ax.tick_params(labelsize=18) 
cb2.ax.yaxis.offsetText.set_fontsize(18)
cb3 = fig.colorbar(pc3,extend='both',fraction=0.05)
cb3.set_label('w$\\cdot$du/dz [1/s]', fontsize=18)
cb3.ax.tick_params(labelsize=18) 
cb3.ax.yaxis.offsetText.set_fontsize(18)
for ax  in axs:
        ax.coastlines()
        ax.add_feature(cart.feature.LAND,facecolor='lightgrey')
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                    linewidth=0, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 20}
        gl.ylabel_style =  {'size': 20}

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection':ccrs.PlateCarree()})
pc1 =ax.pcolormesh(ratio.longitude,ratio.latitude,ratio,vmin=0,vmax=100,cmap='magma')
cb1 = fig.colorbar(pc1,extend='max',fraction=0.05)
cb1.set_label('u$\\cdot$du/dx [1/s]', fontsize=18)
cb1.ax.tick_params(labelsize=18) 
cb1.ax.yaxis.offsetText.set_fontsize(18)


ax.coastlines()
ax.add_feature(cart.feature.LAND,facecolor='lightgrey')
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                linewidth=0, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 20}
gl.ylabel_style =  {'size': 20}

In [ ]:
fig, ax =plt.subplots()
# make pdf out of it?
array = u_dudx.values.flatten()
array = array[~np.isnan(array)]
bins, pdf = make_PDF(array,nbins=500,vmin=-1e-6,vmax=1e-6,norm=True)
ax.plot(bins,pdf,'-',color='navy')

array = v_dudy.values.flatten()
array = array[~np.isnan(array)]
bins, pdf = make_PDF(array,nbins=500,vmin=-1e-6,vmax=1e-6,norm=True)
ax.plot(bins,pdf,'--',color='firebrick')

array = w_dudz.values.flatten()
array = array[~np.isnan(array)]
bins, pdf = make_PDF(array,nbins=500,vmin=-1e-6,vmax=1e-6,norm=True)
ax.plot(bins,pdf,':',color='green')
ax.set_yscale('log')

